# Task 4 - Conditional GAN (Face to Sketch)

End-to-end Colab workflow for Task 4 of the Generative AI assignment.

| Step | What it does |
| --- | --- |
| 1 | Install dependencies, verify GPU |
| 2 | Clone the repo at branch `task4` and enter it |
| 3 | Locate the FS2K dataset and verify its layout |
| 4 | Authenticate Weights & Biases |
| 5 | Optuna hyperparameter search |
| 6 | Full training run |
| 7 | Evaluation with error maps |
| 8 | ONNX export + PyTorch parity check |

Run the cells **top to bottom**. Step 2 changes the working directory to the
cloned repo, which every later step depends on.

## 1. Install dependencies

The install list mirrors `requirements.txt`. `scikit-learn`, `scikit-image`,
`tqdm` and `onnxruntime` are required by the training, evaluation and parity
check scripts respectively, so they are included here.

In [ ]:
!pip install -q torch torchvision optuna wandb matplotlib numpy pyyaml pillow onnx onnxruntime scikit-learn scikit-image tqdm

In [ ]:
import torch

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Training will be extremely slow on CPU.")

## 2. Clone the repository

The repo is **private**, so an authenticated clone is required. The token is read
from Colab Secrets (`userdata`) and never written into the notebook or into the
clone's `.git/config` - it is passed as a transient `extraheader` for the clone
call only.

Add a secret named `GITHUB_TOKEN` (Colab UI: the key icon in the left sidebar ->
"Add new secret"). A classic PAT with `repo` scope is sufficient.

In [ ]:
import base64
import os
import subprocess

REPO_URL = "https://github.com/AKIF-jk/Generative_AI.git"
REPO_DIR = "Generative_AI"
BRANCH = "task4"


def run(cmd, **kwargs):
    result = subprocess.run(cmd, shell=True, text=True,
                            capture_output=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stdout}\n{result.stderr}")
    return result.stdout.strip()


def get_token():
    from google.colab import userdata
    for name in ("GITHUB_TOKEN", "GITHUB_TOKEN_T4"):
        try:
            token = userdata.get(name)
            if token:
                print(f"Using Colab secret '{name}'.")
                return token
        except Exception:
            continue
    raise RuntimeError(
        "No GitHub token found in Colab Secrets.\n"
        "Add one named GITHUB_TOKEN (key icon in the sidebar -> Add new secret)."
    )


if os.path.isdir(REPO_DIR):
    print(f"'{REPO_DIR}' already exists - fetching latest instead of re-cloning.")
    auth = base64.b64encode(f"x-access-token:{get_token()}".encode()).decode()
    run(f'git -C {REPO_DIR} remote set-url origin "{REPO_URL}"')
    run(f'git -c "http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}" '
        f'-C {REPO_DIR} fetch origin')
    run(f'git -C {REPO_DIR} checkout {BRANCH}')
    run(f'git -C {REPO_DIR} reset --hard origin/{BRANCH}')
    del auth
else:
    auth = base64.b64encode(f"x-access-token:{get_token()}".encode()).decode()
    run(f'git -c "http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}" '
        f'clone --branch {BRANCH} --single-branch "{REPO_URL}" {REPO_DIR}')
    del auth

print(run(f'git -C {REPO_DIR} log -1 --oneline'))

In [ ]:
# Enter the repo. %cd persists across cells, unlike `!cd`.
%cd Generative_AI

In [ ]:
import os
import sys

REPO_ROOT = os.getcwd()
SRC = os.path.join(REPO_ROOT, "src", "task4_face2sketch")

print(f"Working directory: {REPO_ROOT}")
print(f"Commit           : {run('git rev-parse --short HEAD')}")

# The Task 4 modules use flat imports (`from data.dataset import ...`), which
# resolve because Python puts the script's own directory on sys.path.
sys.path.insert(0, SRC)

for required in ("models/generator_unet.py", "models/discriminator_patchgan.py",
                 "data/dataset.py", "configs/task4.yaml"):
    path = os.path.join(SRC, required)
    print(f"{'OK  ' if os.path.isfile(path) else 'MISS'} {required}")

## 3. Locate the FS2K dataset

The loaders expect this layout under `data/FS2K`:

```
data/FS2K/
  anno_train.json          # official training split
  anno_test.json           # official test split
  photo/photo1/image0110.jpg
  sketch/sketch1/sketch0110.jpg
  photo/photo2/...   sketch/sketch2/...
  photo/photo3/...   sketch/sketch3/...
```

`anno_train.json` is split 85/15 stratified by style (seed 42) into
train/validation. `anno_test.json` is only read during evaluation.

FS2K is distributed through its authors' project page rather than a stable
direct link, so this cell does not attempt an automatic download. Put the
extracted dataset on Google Drive (or upload the archive to Drive and unzip it
here) and this cell will find and symlink it.

In [ ]:
import json
import os

DATA_DIR = os.path.join(REPO_ROOT, "data", "FS2K")


def verify_fs2k(root):
    problems = []
    for anno in ("anno_train.json", "anno_test.json"):
        if not os.path.isfile(os.path.join(root, anno)):
            problems.append(f"missing {anno}")
    if not os.path.isdir(os.path.join(root, "photo")):
        problems.append("missing photo/ directory")
    if not os.path.isdir(os.path.join(root, "sketch")):
        problems.append("missing sketch/ directory")
    return problems


if not os.path.isdir(DATA_DIR):
    from google.colab import drive
    drive.mount("/content/drive")

    candidates = [
        "/content/drive/MyDrive/FS2K",
        "/content/drive/MyDrive/data/FS2K",
        "/content/drive/MyDrive/Generative_AI/data/FS2K",
    ]
    linked = False
    for candidate in candidates:
        if verify_fs2k(candidate) == []:
            os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
            os.symlink(candidate, DATA_DIR)
            print(f"Linked {DATA_DIR} -> {candidate}")
            linked = True
            break

    if not linked:
        listing = []
        for root, dirs, _ in os.walk("/content/drive/MyDrive"):
            depth = root[len("/content/drive/MyDrive"):].count(os.sep)
            if depth > 2:
                dirs[:] = []
                continue
            if "FS2K" in dirs or os.path.basename(root) == "FS2K":
                listing.append(root)
        raise FileNotFoundError(
            "Could not locate FS2K on Drive.\n"
            "Upload/extract the dataset so that it contains anno_train.json, "
            "anno_test.json, photo/ and sketch/.\n"
            f"Candidates checked: {candidates}\n"
            f"FS2K-like dirs found on Drive: {listing[:10]}"
        )
else:
    print(f"Using existing dataset at {DATA_DIR}")

problems = verify_fs2k(DATA_DIR)
if problems:
    raise FileNotFoundError(f"Incomplete dataset at {DATA_DIR}: {problems}")

In [ ]:
def summarise(split):
    with open(os.path.join(DATA_DIR, f"anno_{split}.json")) as f:
        anno = json.load(f)
    counts = {}
    for item in anno:
        counts[item["style"]] = counts.get(item["style"], 0) + 1
    print(f"{split:5s}: {len(anno):4d} pairs  |  "
          + "  ".join(f"style {s}: {counts.get(s, 0)}" for s in sorted(counts)))
    return len(anno)


n_train = summarise("train")
n_test = summarise("test")
print(f"\nValidation is a 15% stratified split of the {n_train} training pairs "
      f"(~{round(n_train * 0.15)} pairs, seed 42).")
print(f"The {n_test} test pairs are never seen during training or HPO.")

## 4. Weights & Biases

Training calls `wandb.init(...)` unconditionally. Authenticate with
`wandb login`, or leave `WANDB_MODE` as `offline` to log locally instead
(runs land in `./wandb/`; run `wandb sync wandb/offline-runs-*` later).

In [ ]:
import os

os.environ.setdefault("WANDB_MODE", "offline")
print(f"WANDB_MODE={os.environ['WANDB_MODE']}")

# To log to the cloud instead, store a key in Colab Secrets as WANDB_API_KEY
# and replace the line above with:
#   os.environ["WANDB_MODE"] = "online"
#   from google.colab import userdata
#   os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# Or just run:  !wandb login <your-api-key>

## 5. Optuna hyperparameter search

Searches `lr_g`, `lr_d`, `batch_size`, `base_channels`, `dropout`, `style_dim`
and `lambda_l1` over 15 trials with a median pruner, using a reduced 5-epoch
budget. The objective is validation L1 (lower is better).

In [ ]:
!python src/task4_face2sketch/optuna_search.py

## 6. Full training

Runs the schedule in `src/task4_face2sketch/configs/task4.yaml`.

Note that `evaluate.py` and `export_onnx.py` load the checkpoint assuming
`base_channels=64` and `style_dim=16`. If the Optuna winner differs, edit those
defaults to match before running steps 7 and 8.

In [ ]:
!python src/task4_face2sketch/train.py --config src/task4_face2sketch/configs/task4.yaml

In [ ]:
import os
from IPython.display import Image, display

checkpoint = os.path.join(REPO_ROOT, "best_generator.pth")
size_mb = os.path.getsize(checkpoint) / 1e6 if os.path.exists(checkpoint) else 0
print(f"best_generator.pth: {'present' if size_mb else 'MISSING'} ({size_mb:.1f} MB)")

sample = "evaluation_results/eval_sample_000_style_1.png"
if os.path.isfile(sample):
    display(Image(filename=sample))

## 7. Evaluation

Evaluates `best_generator.pth` on the held-out test split, reporting per-style
and overall L1 and SSIM, and writes photo / target / generated / error-map
panels into `evaluation_results/`.

In [ ]:
!python src/task4_face2sketch/evaluate.py

## 8. ONNX export

Exports the generator to `generator.onnx` (photo + style_id inputs, dynamic
batch axis) and asserts that ONNX Runtime and PyTorch agree to within `1e-4`.

In [ ]:
!python src/task4_face2sketch/export_onnx.py

In [ ]:
import os

onnx_path = os.path.join(REPO_ROOT, "generator.onnx")
if os.path.isfile(onnx_path):
    print(f"generator.onnx: {os.path.getsize(onnx_path) / 1e6:.1f} MB")
else:
    print("generator.onnx: MISSING")